# 02 — Modelo Logístico WOE Final

Este notebook contiene el **flujo final, limpio y reproducible** del modelo seleccionado para el Reto Compartamos.

Su objetivo no es repetir todo el análisis exploratorio del notebook de desarrollo, sino ejecutar únicamente las decisiones que ya fueron justificadas y congeladas.

## Decisiones ya cerradas

- Modelo principal: **Regresión Logística + WOE**
- Variables finales: **18**
- Bins aprendidos únicamente en **TRAIN**
- Valores centinela tratados como estados especiales
- Coarse classing final en 2 variables
- Cutoff operativo: **PD < 20%**
- Regla metodológica:

> **TRAIN aprende → VALIDATION decide → OOT confirma**

## ¿Por qué este modelo?

La elección no se hizo únicamente por maximizar Gini. Se priorizó una solución que combine:

- discriminación estable;
- explicabilidad;
- auditabilidad;
- facilidad de implementación;
- facilidad de monitoreo;
- menor complejidad operativa.

El benchmark CatBoost se desarrolla por separado.


## 1. Librerías y configuración

La ruta asume que este notebook se encuentra dentro de una carpeta `notebooks/` y que la base está en:

`data/raw/base_analytics_lab_061628.parquet`

Si tu estructura local es diferente, solo debes cambiar `DATA_PATH`.


In [1]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
import statsmodels.api as sm
import scorecardpy as sc

from scipy.stats import norm
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss


# ============================================================
# RUTAS
# ============================================================

ROOT = Path.cwd().resolve()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "raw" / "base_analytics_lab_061628.parquet"

MODEL_DIR = ROOT / "model"
CONFIG_DIR = ROOT / "config"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

TARGET = "TARGET"
TIEMPO = "CODMES"

CUTOFF_PD_FINAL = 0.20

print("Ruta de datos:", DATA_PATH)
print("Cutoff final:", f"PD < {int(CUTOFF_PD_FINAL * 100)}%")


Ruta de datos: C:\Users\Manolo\Desktop\Compartamos definitivo\Entrega_Compartamos_Final\data\raw\base_analytics_lab_061628.parquet
Cutoff final: PD < 20%


## 2. Carga de datos y validaciones mínimas

En esta versión final no repetimos el EDA completo.

Solo comprobamos que:

- la base pueda cargarse;
- existan las columnas necesarias;
- el TARGET y la variable temporal estén disponibles.


In [2]:
# ============================================================
# CARGA
# ============================================================

df = pd.read_parquet(DATA_PATH).copy()

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

display(df.head())


Filas: 60000
Columnas: 49


,CODMES,TARGET,MEDIDA1_MORA_INTERNA_U6M,MEDIDA1_MORA_INTERNA_U12M,MEDIDA1_MORA_INTERNA_U1M,MEDIDA2_MORA_INTERNA_U12M,MEDIDA2_MORA_INTERNA_U6M,MEDIDA3_MORA_INTERNA_U12M,MEDIDA1_APALANCAMIENTO_INTERNO_U6M,MEDIDA2_APALANCAMIENTO_INTERNO_U6M,...,MEDIDA2_PRODUCTOS_SSFF_U3M,MEDIDA1_LINEA_TARJETACREDITO,PREDICCION_MODELO_ANTERIOR,PROXY_TARGET_6M,MEDIDA1_TRX_CASHIN_U3M,MEDIDA2_TRX_TRANSFER_U1M,MEDIDA3_TRX_QR_U1M,MEDIDA4_TRX_DEBITO_U1M,MEDIDA5_TRX_ECOM_U1M,MEDIDA6_TRX_APP_U1M
0,202308,0,0,0,-1,99,99,0,0.853261,1.000000,...,2.0,NaN,0.250741,0,26.740000,NaN,NaN,1.0,51.860001,1
1,202401,1,4,4,1,99,0,5,0.706940,0.897245,...,5.0,NaN,0.605026,0,71.269997,89.379997,NaN,6.0,49.099998,1
2,202402,0,3,3,3,99,0,6,0.392161,0.869825,...,2.0,NaN,0.601767,0,40.180000,136.169998,NaN,1.0,153.330002,1
3,202307,1,21,21,21,0,0,2,0.706261,1.049905,...,3.0,NaN,0.665567,0,739.659973,NaN,NaN,2.0,176.559998,0
4,202307,1,0,0,0,99,99,0,1.000000,1.238984,...,4.0,NaN,0.564676,1,374.079987,291.910004,NaN,3.0,59.529999,0


## 3. Variables finales

Estas 18 variables son el resultado del proceso completo documentado en `01_desarrollo_modelo_logistico.ipynb`.

La reducción final se realizó mediante Drop-One utilizando VALIDATION.

No se seleccionaron variables mirando OOT.


In [3]:
# ============================================================
# 18 VARIABLES FINALES
# ============================================================

variables_finales_18 = [
    "MEDIDA1_MORA_INTERNA_U6M",
    "MEDIDA1_MORA_INTERNA_U1M",
    "MEDIDA2_MORA_INTERNA_U6M",
    "MEDIDA1_APALANCAMIENTO_INTERNO_U6M",
    "SOCIODEMOGRAFICO_1",
    "MEDIDA3_APALANCAMIENTO_INTERNO_U1M",
    "MEDIDA1_MORA_SSFF_U36M",
    "MEDIDA1_APALANCAMIENTO_SSFF_U36M",
    "MEDIDA1_DESAPALANCAMIENTO_SSFF_U12M",
    "MEDIDA3_APALANCAMIENTO_SSFF_U12M",
    "MEDIDA1_PRODUCTOS_SSFF_U1M",
    "DEUDA_PROMEDIO_SSFF",
    "MEDIDA1_CALIFICACION_SSFF_U60M",
    "MEDIDA2_CALIFICACION_SSFF_U60M",
    "MEDIDA1_DESAPALANCAMIENTO_INTERNO_U12M",
    "MEDIDA1_ANTIGUEDAD_SSFF",
    "MEDIDA1_LINEA_TARJETACREDITO",
    "MEDIDA1_TRX_CASHIN_U3M"
]


columnas_necesarias = variables_finales_18 + [TARGET, TIEMPO]

columnas_faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in df.columns
]

if columnas_faltantes:
    raise ValueError(
        f"Faltan columnas necesarias: {columnas_faltantes}"
    )

print("Número de variables finales:", len(variables_finales_18))
print("Validación de columnas: OK")


Número de variables finales: 18
Validación de columnas: OK


## 4. Partición temporal

Se conserva exactamente la partición definida durante el desarrollo:

- TRAIN: hasta `202401`
- VALIDATION: `202402–202403`
- OOT: `202404–202405`

La separación temporal simula el uso real del modelo sobre períodos posteriores.


In [4]:
# ============================================================
# SPLIT TEMPORAL
# ============================================================

train = df[
    df[TIEMPO] <= 202401
].copy()

validation = df[
    df[TIEMPO].isin([202402, 202403])
].copy()

oot = df[
    df[TIEMPO].isin([202404, 202405])
].copy()


resumen_split = pd.DataFrame({

    "Muestra": [
        "TRAIN",
        "VALIDATION",
        "OOT"
    ],

    "Filas": [
        len(train),
        len(validation),
        len(oot)
    ],

    "Bad_Rate_%": [
        train[TARGET].mean() * 100,
        validation[TARGET].mean() * 100,
        oot[TARGET].mean() * 100
    ]
})


display(
    resumen_split.round(2)
)


,Muestra,Filas,Bad_Rate_%
0,TRAIN,38462,16.02
1,VALIDATION,10583,14.95
2,OOT,10955,14.13


## 5. Valores centinela

Los valores especiales no se reemplazan automáticamente por `NaN`.

Durante el desarrollo se comprobó que algunos códigos representan **estados de negocio** con un comportamiento de riesgo propio.

Por ello se conservan como bins independientes.


In [5]:
# ============================================================
# VALORES ESPECIALES UTILIZADOS EN LAS 18 VARIABLES FINALES
# ============================================================

valores_especiales = {

    "MEDIDA2_MORA_INTERNA_U6M": [99],

    "MEDIDA3_APALANCAMIENTO_INTERNO_U1M": [
        333333344
    ],

    "MEDIDA1_MORA_SSFF_U36M": [99],

    "MEDIDA2_CALIFICACION_SSFF_U60M": [60]
}


for variable, valores in valores_especiales.items():
    print(variable, "→", valores)


MEDIDA2_MORA_INTERNA_U6M → [99]
MEDIDA3_APALANCAMIENTO_INTERNO_U1M → [333333344]
MEDIDA1_MORA_SSFF_U36M → [99]
MEDIDA2_CALIFICACION_SSFF_U60M → [60]


## 6. Binning WOE final

El WOE transforma los valores originales en grupos con comportamiento de riesgo homogéneo.

Los bins se aprenden **solo con TRAIN**.

Luego exactamente esos mismos bins se aplican a VALIDATION y OOT.

Esto evita leakage.

### Coarse classing congelado

Durante el desarrollo se decidió ajustar manualmente dos variables por interpretabilidad y estabilidad:

- `MEDIDA1_LINEA_TARJETACREDITO`: `<5000` y `5000+`
- `MEDIDA2_MORA_INTERNA_U6M`: `<1`, `1–3`, `3+`, manteniendo `99` como bin especial


In [6]:
# ============================================================
# DATAFRAME DE TRAIN PARA WOE
# ============================================================

train_sc = train[
    variables_finales_18 + [TARGET]
].copy()

train_sc[TARGET] = (
    train_sc[TARGET]
    .astype("int64")
)


# ============================================================
# BINNING AUTOMÁTICO
# ============================================================

bins_woe_final = sc.woebin(

    train_sc,

    y=TARGET,

    x=variables_finales_18,

    positive="1",

    special_values=valores_especiales,

    method="tree",

    bin_num_limit=8,

    count_distr_limit=0.05,

    no_cores=1,

    print_step=0
)


# ============================================================
# COARSE CLASSING 1
# LINEA DE TARJETA
# ============================================================

bin_linea_ajustado = sc.woebin(

    train_sc,

    y=TARGET,

    x=["MEDIDA1_LINEA_TARJETACREDITO"],

    positive="1",

    breaks_list={
        "MEDIDA1_LINEA_TARJETACREDITO": [5000]
    },

    no_cores=1,

    print_step=0
)


# ============================================================
# COARSE CLASSING 2
# MORA INTERNA U6M
# ============================================================

bin_mora_ajustado = sc.woebin(

    train_sc,

    y=TARGET,

    x=["MEDIDA2_MORA_INTERNA_U6M"],

    positive="1",

    breaks_list={
        "MEDIDA2_MORA_INTERNA_U6M": [1, 3]
    },

    special_values={
        "MEDIDA2_MORA_INTERNA_U6M": [99]
    },

    no_cores=1,

    print_step=0
)


# Reemplazamos solamente esas dos definiciones
bins_woe_final[
    "MEDIDA1_LINEA_TARJETACREDITO"
] = bin_linea_ajustado[
    "MEDIDA1_LINEA_TARJETACREDITO"
]

bins_woe_final[
    "MEDIDA2_MORA_INTERNA_U6M"
] = bin_mora_ajustado[
    "MEDIDA2_MORA_INTERNA_U6M"
]


print("Binning final construido con TRAIN.")
print("Variables con bins:", len(bins_woe_final))


[INFO] creating woe binning ...


c:\Users\Manolo\AppData\Local\Programs\Python\Python311\Lib\site-packages\scorecardpy\condition_fun.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  datetime_cols = dat.apply(pd.to_numeric,errors='ignore').select_dtypes(object).apply(pd.to_datetime,errors='ignore').select_dtypes('datetime64').columns.tolist()
c:\Users\Manolo\AppData\Local\Programs\Python\Python311\Lib\site-packages\scorecardpy\condition_fun.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  datetime_cols = dat.apply(pd.to_numeric,errors='ignore').select_dtypes(object).apply(pd.to_datetime,errors='ignore').select_dtypes('datetime64').columns.tolist()
c:\Users\Manolo\AppData\Local\Programs\Python\Python311\Lib\site-packages\scorecardpy\woebin.py:320: FutureWarning: The default of obse

[INFO] creating woe binning ...


c:\Users\Manolo\AppData\Local\Programs\Python\Python311\Lib\site-packages\scorecardpy\condition_fun.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  datetime_cols = dat.apply(pd.to_numeric,errors='ignore').select_dtypes(object).apply(pd.to_datetime,errors='ignore').select_dtypes('datetime64').columns.tolist()
c:\Users\Manolo\AppData\Local\Programs\Python\Python311\Lib\site-packages\scorecardpy\condition_fun.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  datetime_cols = dat.apply(pd.to_numeric,errors='ignore').select_dtypes(object).apply(pd.to_datetime,errors='ignore').select_dtypes('datetime64').columns.tolist()
c:\Users\Manolo\AppData\Local\Programs\Python\Python311\Lib\site-packages\scorecardpy\woebin.py:38: FutureWarning: Downcasting behavior

[INFO] creating woe binning ...
Binning final construido con TRAIN.
Variables con bins: 18


c:\Users\Manolo\AppData\Local\Programs\Python\Python311\Lib\site-packages\scorecardpy\condition_fun.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  datetime_cols = dat.apply(pd.to_numeric,errors='ignore').select_dtypes(object).apply(pd.to_datetime,errors='ignore').select_dtypes('datetime64').columns.tolist()
c:\Users\Manolo\AppData\Local\Programs\Python\Python311\Lib\site-packages\scorecardpy\condition_fun.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  datetime_cols = dat.apply(pd.to_numeric,errors='ignore').select_dtypes(object).apply(pd.to_datetime,errors='ignore').select_dtypes('datetime64').columns.tolist()
c:\Users\Manolo\AppData\Local\Programs\Python\Python311\Lib\site-packages\scorecardpy\woebin.py:141: FutureWarning: The provided callab

## 7. Aplicar la transformación WOE

La misma función se utiliza en las tres muestras.

No se recalculan bins en VALIDATION ni OOT.


In [7]:
def transformar_woe(base):

    datos = base[
        variables_finales_18 + [TARGET]
    ].copy()

    datos[TARGET] = (
        datos[TARGET]
        .astype("int64")
    )

    datos_woe = sc.woebin_ply(
        datos,
        bins_woe_final,
        print_step=0
    )

    return datos_woe


train_woe = transformar_woe(train)

validation_woe = transformar_woe(validation)

oot_woe = transformar_woe(oot)


print("TRAIN WOE:", train_woe.shape)
print("VALIDATION WOE:", validation_woe.shape)
print("OOT WOE:", oot_woe.shape)


[INFO] converting into woe values ...
[INFO] converting into woe values ...
[INFO] converting into woe values ...
TRAIN WOE: (38462, 19)
VALIDATION WOE: (10583, 19)
OOT WOE: (10955, 19)


## 8. Entrenamiento de la Regresión Logística final

La regresión se estima únicamente con TRAIN.

Las variables utilizadas por el modelo son las 18 columnas WOE.


In [8]:
# ============================================================
# PREPARAR MATRICES
# ============================================================

variables_woe = [
    variable + "_woe"
    for variable in variables_finales_18
]


X_train = sm.add_constant(
    train_woe[variables_woe].copy()
)

X_valid = sm.add_constant(
    validation_woe[variables_woe].copy(),
    has_constant="add"
)

X_oot = sm.add_constant(
    oot_woe[variables_woe].copy(),
    has_constant="add"
)


y_train = train_woe[TARGET]
y_valid = validation_woe[TARGET]
y_oot = oot_woe[TARGET]


# ============================================================
# ENTRENAMIENTO FINAL
# ============================================================

modelo_final = sm.Logit(
    y_train,
    X_train
).fit(
    disp=False
)


tabla_coeficientes = pd.DataFrame({

    "Variable": modelo_final.params.index,

    "Coeficiente": modelo_final.params.values,

    "p_value": modelo_final.pvalues.values
})


tabla_coeficientes["Significativa_5%"] = (
    tabla_coeficientes["p_value"] < 0.05
)


display(
    tabla_coeficientes.round(6)
)


,Variable,Coeficiente,p_value,Significativa_5%
0,const,-1.685968,0.0,True
1,MEDIDA1_MORA_INTERNA_U6M_woe,0.721606,0.0,True
2,MEDIDA1_MORA_INTERNA_U1M_woe,0.396254,0.0,True
3,MEDIDA2_MORA_INTERNA_U6M_woe,0.400583,0.0,True
4,MEDIDA1_APALANCAMIENTO_INTERNO_U6M_woe,0.915560,0.0,True
5,SOCIODEMOGRAFICO_1_woe,0.879287,0.0,True
6,MEDIDA3_APALANCAMIENTO_INTERNO_U1M_woe,0.799309,0.0,True
7,MEDIDA1_MORA_SSFF_U36M_woe,0.518686,0.0,True
8,MEDIDA1_APALANCAMIENTO_SSFF_U36M_woe,0.546743,0.0,True
9,MEDIDA1_DESAPALANCAMIENTO_SSFF_U12M_woe,0.405430,0.0,True


## 9. Performance temporal

La evaluación se realiza con AUC, Gini y KS.

El objetivo no es únicamente obtener una métrica alta, sino comprobar que el desempeño sea razonablemente estable al avanzar en el tiempo.


In [9]:
# ============================================================
# FUNCIONES DE MÉTRICAS
# ============================================================

def calcular_ks(y_real, probabilidades):

    fpr, tpr, _ = roc_curve(
        y_real,
        probabilidades
    )

    return np.max(
        tpr - fpr
    )


def resumen_metricas(
    nombre,
    y_real,
    probabilidades
):

    auc = roc_auc_score(
        y_real,
        probabilidades
    )

    return {

        "Muestra": nombre,

        "AUC": auc,

        "Gini": 2 * auc - 1,

        "KS": calcular_ks(
            y_real,
            probabilidades
        ),

        "Bad_Rate_%": (
            np.mean(y_real) * 100
        ),

        "PD_Promedio_%": (
            np.mean(probabilidades) * 100
        ),

        "Brier": brier_score_loss(
            y_real,
            probabilidades
        )
    }


# ============================================================
# PREDICCIONES
# ============================================================

prob_train = modelo_final.predict(
    X_train
)

prob_valid = modelo_final.predict(
    X_valid
)

prob_oot = modelo_final.predict(
    X_oot
)


tabla_metricas = pd.DataFrame([

    resumen_metricas(
        "TRAIN",
        y_train,
        prob_train
    ),

    resumen_metricas(
        "VALIDATION",
        y_valid,
        prob_valid
    ),

    resumen_metricas(
        "OOT",
        y_oot,
        prob_oot
    )
])


display(
    tabla_metricas.round(6)
)


,Muestra,AUC,Gini,KS,Bad_Rate_%,PD_Promedio_%,Brier
0,TRAIN,0.799507,0.599015,0.445362,16.018408,16.018408,0.106637
1,VALIDATION,0.790788,0.581576,0.432179,14.948502,15.875251,0.103930
2,OOT,0.796370,0.592741,0.449504,14.130534,16.538921,0.099094


### Valores de referencia esperados

Si el entorno y las versiones de librerías reproducen el desarrollo, los resultados deberían quedar aproximadamente alrededor de:

- TRAIN Gini ≈ `0.599`
- VALIDATION Gini ≈ `0.582`
- OOT Gini ≈ `0.593`

Pequeñas diferencias numéricas deben revisarse, pero no deberían existir cambios materiales.


## 10. Calibración general

Aquí comparamos la PD promedio con el Bad Rate observado.

Esto mide una propiedad diferente de la discriminación:

- **discriminación:** ordenar correctamente clientes más y menos riesgosos;
- **calibración:** que el nivel absoluto de PD sea coherente con el default observado.


In [10]:
tabla_calibracion = pd.DataFrame({

    "Muestra": [
        "TRAIN",
        "VALIDATION",
        "OOT"
    ],

    "PD_Promedio_%": [
        np.mean(prob_train) * 100,
        np.mean(prob_valid) * 100,
        np.mean(prob_oot) * 100
    ],

    "Bad_Rate_Real_%": [
        np.mean(y_train) * 100,
        np.mean(y_valid) * 100,
        np.mean(y_oot) * 100
    ]
})


tabla_calibracion["Gap_pp"] = (
    tabla_calibracion["PD_Promedio_%"]
    - tabla_calibracion["Bad_Rate_Real_%"]
)


display(
    tabla_calibracion.round(4)
)


,Muestra,PD_Promedio_%,Bad_Rate_Real_%,Gap_pp
0,TRAIN,16.0184,16.0184,-0.0000
1,VALIDATION,15.8753,14.9485,0.9267
2,OOT,16.5389,14.1305,2.4084


## 11. Backtesting de PD por bandas — VALIDATION

El desarrollo completo mostró:

- 6 de 7 bandas no presentan una diferencia estadísticamente significativa bajo independencia;
- la banda `20%-30%` presentó una sobreestimación estadísticamente significativa;
- al incorporar sensibilidad Vasicek con `rho` entre 0.5% y 3%, las 7 bandas resultaron compatibles.

En este notebook final reproducimos el backtesting base bajo independencia.


In [11]:
# ============================================================
# BANDAS DE PD
# ============================================================

bins_pd = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.30,
    0.40,
    1.00
]

labels_pd = [
    "0%-5%",
    "5%-10%",
    "10%-15%",
    "15%-20%",
    "20%-30%",
    "30%-40%",
    "40%+"
]


backtest_valid = pd.DataFrame({

    "TARGET": np.asarray(
        y_valid
    ).astype(int),

    "PD": np.asarray(
        prob_valid
    ).astype(float)
})


backtest_valid["BANDA_PD"] = pd.cut(

    backtest_valid["PD"],

    bins=bins_pd,

    labels=labels_pd,

    include_lowest=True,

    right=False
)


resultados_backtest = []


for banda, grupo in backtest_valid.groupby(
    "BANDA_PD",
    observed=True
):

    n = len(grupo)

    observados = int(
        grupo["TARGET"].sum()
    )

    probabilidades = (
        grupo["PD"].values
    )

    esperados = (
        probabilidades.sum()
    )

    varianza = (
        probabilidades
        * (1 - probabilidades)
    ).sum()

    desviacion = np.sqrt(
        varianza
    )

    if desviacion > 0:

        z = (
            observados
            - esperados
        ) / desviacion

        p_value = 2 * (
            1 - norm.cdf(
                abs(z)
            )
        )

    else:

        z = np.nan
        p_value = np.nan


    resultados_backtest.append({

        "Banda_PD": str(banda),

        "Clientes": n,

        "PD_Promedio_%": (
            probabilidades.mean()
            * 100
        ),

        "Bad_Rate_Real_%": (
            observados / n
            * 100
        ),

        "Gap_pp": (
            probabilidades.mean()
            - observados / n
        ) * 100,

        "p_value": p_value,

        "Resultado_5%": (
            "Sin diferencia estadísticamente significativa"
            if pd.notna(p_value)
            and p_value >= 0.05
            else "Diferencia estadísticamente significativa"
        )
    })


tabla_backtesting_valid = pd.DataFrame(
    resultados_backtest
)


display(
    tabla_backtesting_valid.round(4)
)


,Banda_PD,Clientes,PD_Promedio_%,Bad_Rate_Real_%,Gap_pp,p_value,Resultado_5%
0,0%-5%,2655,2.7265,2.9379,-0.2113,0.5023,Sin diferencia estadísticamente significativa
1,5%-10%,2343,7.3944,6.5728,0.8217,0.1280,Sin diferencia estadísticamente significativa
2,10%-15%,1718,12.3804,11.4668,0.9136,0.2498,Sin diferencia estadísticamente significativa
3,15%-20%,1175,17.3540,17.3617,-0.0077,0.9944,Sin diferencia estadísticamente significativa
4,20%-30%,1193,24.1481,20.7041,3.4440,0.0053,Diferencia estadísticamente significativa
5,30%-40%,581,34.5475,34.0792,0.4684,0.8120,Sin diferencia estadísticamente significativa
6,40%+,918,57.6278,54.9020,2.7259,0.0817,Sin diferencia estadísticamente significativa


## 12. Política de renovación

La política se comunica directamente mediante PD.

Durante el desarrollo se compararon distintos cutoffs enteros en VALIDATION.

La regla seleccionada fue:

> **Habilitar renovación si PD < 20%**

En VALIDATION produjo aproximadamente:

- 74.56% de aprobación;
- 8.02% de Bad Rate entre aprobados.

Después el cutoff se congeló.


In [12]:
def evaluar_politica(
    y_real,
    probabilidades,
    cutoff
):

    y = np.asarray(
        y_real
    ).astype(int)

    pd_modelo = np.asarray(
        probabilidades
    ).astype(float)

    aprobados = (
        pd_modelo < cutoff
    )

    rechazados = ~aprobados

    malos_totales = (
        y.sum()
    )

    return {

        "Regla": (
            f"PD < {int(cutoff * 100)}%"
        ),

        "Aprobacion_%": (
            aprobados.mean() * 100
        ),

        "Rechazo_%": (
            rechazados.mean() * 100
        ),

        "Bad_Rate_Aprobados_%": (
            y[aprobados].mean() * 100
        ),

        "Malos_Rechazados_%": (
            y[rechazados].sum()
            / malos_totales
            * 100
        )
    }


tabla_politica_final = pd.DataFrame([

    {
        "Muestra": "VALIDATION",
        **evaluar_politica(
            y_valid,
            prob_valid,
            CUTOFF_PD_FINAL
        )
    },

    {
        "Muestra": "OOT",
        **evaluar_politica(
            y_oot,
            prob_oot,
            CUTOFF_PD_FINAL
        )
    }
])


display(
    tabla_politica_final.round(2)
)


,Muestra,Regla,Aprobacion_%,Rechazo_%,Bad_Rate_Aprobados_%,Malos_Rechazados_%
0,VALIDATION,PD < 20%,74.56,25.44,8.02,59.99
1,OOT,PD < 20%,73.47,26.53,6.96,63.82


### Resultado esperado de la política

La regla fue escogida en VALIDATION y **no se modifica con OOT**.

Resultados esperados:

| Muestra | Aprobación | Bad Rate aprobados | Malos rechazados |
|---|---:|---:|---:|
| VALIDATION | 74.56% | 8.02% | 59.99% |
| OOT | 73.47% | 6.96% | 63.82% |

Esto conecta directamente el modelo con el objetivo de negocio.


## 12.1 Análisis de robustez del cutoff

El cutoff `PD < 20%` no debe interpretarse como un punto aislado elegido porque produce exactamente 8% en VALIDATION.

Para comprobar que la política no es excesivamente sensible al umbral, se evalúan cutoffs cercanos entre 17% y 23%.

El objetivo es verificar que:

- la aprobación cambie de forma gradual;
- el Bad Rate de aprobados cambie de forma gradual;
- no existan saltos abruptos alrededor del 20%;
- la conclusión se mantenga razonablemente estable en OOT.


In [13]:
cutoffs_robustez = [
    0.17,
    0.18,
    0.19,
    0.20,
    0.21,
    0.22,
    0.23
]

tabla_robustez_valid = pd.DataFrame([
    {
        "Muestra": "VALIDATION",
        "Cutoff_PD_%": cutoff * 100,
        **evaluar_politica(
            y_valid,
            prob_valid,
            cutoff
        )
    }
    for cutoff in cutoffs_robustez
])

tabla_robustez_oot = pd.DataFrame([
    {
        "Muestra": "OOT",
        "Cutoff_PD_%": cutoff * 100,
        **evaluar_politica(
            y_oot,
            prob_oot,
            cutoff
        )
    }
    for cutoff in cutoffs_robustez
])

tabla_robustez = pd.concat(
    [
        tabla_robustez_valid,
        tabla_robustez_oot
    ],
    ignore_index=True
)

display(
    tabla_robustez.round(2)
)


,Muestra,Cutoff_PD_%,Regla,Aprobacion_%,Rechazo_%,Bad_Rate_Aprobados_%,Malos_Rechazados_%
0,VALIDATION,17.0,PD < 17%,68.42,31.58,7.02,67.89
1,VALIDATION,18.0,PD < 18%,70.59,29.41,7.28,65.61
2,VALIDATION,19.0,PD < 19%,72.66,27.34,7.67,62.71
3,VALIDATION,20.0,PD < 20%,74.56,25.44,8.02,59.99
4,VALIDATION,21.0,PD < 21%,76.31,23.69,8.26,57.84
5,VALIDATION,22.0,PD < 22%,77.99,22.01,8.52,55.56
6,VALIDATION,23.0,PD < 23%,79.36,20.64,8.79,53.35
7,OOT,17.0,PD < 17%,66.68,33.32,6.02,71.58
8,OOT,18.0,PD < 18%,69.16,30.84,6.38,68.80
9,OOT,19.0,PD < 19%,71.41,28.59,6.62,66.54


### Resultado del análisis de robustez

El comportamiento alrededor del cutoff seleccionado es gradual.

En VALIDATION, el Bad Rate de aprobados evoluciona aproximadamente:

- PD < 17% → 7.02%
- PD < 18% → 7.28%
- PD < 19% → 7.67%
- **PD < 20% → 8.02%**
- PD < 21% → 8.26%
- PD < 22% → 8.52%
- PD < 23% → 8.79%

En OOT:

- PD < 17% → 6.02%
- PD < 18% → 6.38%
- PD < 19% → 6.62%
- **PD < 20% → 6.96%**
- PD < 21% → 7.28%
- PD < 22% → 7.48%
- PD < 23% → 7.67%

No se observan saltos abruptos alrededor del 20%.

Por ello, `PD < 20%` se encuentra dentro de una **zona de decisión estable** y no corresponde a un punto accidental.

La regla mantiene además un comportamiento consistente fuera de tiempo, sin necesidad de reajustar el cutoff utilizando OOT.


## 13. Guardar un único artefacto del modelo

El entregable de producción debe conservar juntos:

- modelo logístico;
- bins WOE;
- lista y orden de variables;
- valores centinela;
- coarse classing;
- cutoff de PD;
- metadatos básicos.

Así evitamos que el modelo y su transformación viajen por separado.


In [14]:
# ============================================================
# ARTEFACTO FINAL
# ============================================================

artefacto_modelo = {

    "model": modelo_final,

    "bins_woe": bins_woe_final,

    "variables_finales": variables_finales_18,

    "variables_woe": variables_woe,

    "special_values": valores_especiales,

    "cutoff_pd": CUTOFF_PD_FINAL,

    "target": TARGET,

    "time_column": TIEMPO,

    "split": {
        "train_max": 202401,
        "validation": [
            202402,
            202403
        ],
        "oot": [
            202404,
            202405
        ]
    },

    "model_type": (
        "Regresion Logistica + WOE"
    )
}


MODEL_PATH = (
    MODEL_DIR
    / "modelo_logistico_woe.pkl"
)


joblib.dump(
    artefacto_modelo,
    MODEL_PATH
)


print(
    "Modelo guardado en:",
    MODEL_PATH
)


Modelo guardado en: C:\Users\Manolo\Desktop\Compartamos definitivo\Entrega_Compartamos_Final\model\modelo_logistico_woe.pkl


## 14. Función de predicción

La función recibe una base nueva con las 18 variables originales y devuelve:

- PD estimada;
- decisión de renovación según `PD < 20%`.

En producción, el TARGET no estará disponible ni es necesario para predecir.


In [15]:
def predecir_pd(
    base_nueva,
    artefacto
):

    variables = (
        artefacto[
            "variables_finales"
        ]
    )

    variables_woe_local = (
        artefacto[
            "variables_woe"
        ]
    )

    bins = (
        artefacto[
            "bins_woe"
        ]
    )

    modelo = (
        artefacto[
            "model"
        ]
    )

    cutoff = (
        artefacto[
            "cutoff_pd"
        ]
    )


    faltantes = [
        variable
        for variable in variables
        if variable not in base_nueva.columns
    ]

    if faltantes:

        raise ValueError(
            f"Faltan variables: {faltantes}"
        )


    # ScorecardPy necesita solamente
    # las variables que transformará.
    datos = (
        base_nueva[
            variables
        ]
        .copy()
    )


    datos_woe = sc.woebin_ply(
        datos,
        bins,
        print_step=0
    )


    X = sm.add_constant(
        datos_woe[
            variables_woe_local
        ].copy(),
        has_constant="add"
    )


    pd_estimada = (
        modelo.predict(X)
    )


    resultado = pd.DataFrame({

        "PD": pd_estimada,

        "PD_%": (
            pd_estimada * 100
        ),

        "HABILITAR_RENOVACION": (
            pd_estimada < cutoff
        )
    })


    return resultado


## 15. Prueba de reproducibilidad

Guardamos y volvemos a cargar el `.pkl`.

Luego comprobamos que las predicciones obtenidas desde el artefacto coincidan con las PD calculadas anteriormente.

La diferencia debe ser prácticamente cero.


In [16]:
# ============================================================
# RECARGAR ARTEFACTO
# ============================================================

artefacto_cargado = joblib.load(
    MODEL_PATH
)


# Probamos sobre las primeras 100 filas de OOT
muestra_prueba = (
    oot.head(100).copy()
)


prediccion_recargada = predecir_pd(
    muestra_prueba,
    artefacto_cargado
)


pd_original = np.asarray(
    prob_oot[:100]
)


diferencia_maxima = np.max(
    np.abs(
        pd_original
        - prediccion_recargada[
            "PD"
        ].values
    )
)


print(
    "Diferencia máxima:",
    diferencia_maxima
)


display(
    prediccion_recargada.head()
)


[INFO] converting into woe values ...
Diferencia máxima: 0.0


,PD,PD_%,HABILITAR_RENOVACION
5,0.069427,6.942737,True
10,0.011748,1.174822,True
25,0.114600,11.459971,True
28,0.148024,14.802351,True
30,0.061386,6.138614,True


## 16. Cierre

Este notebook representa el flujo operativo final:

**base original → split temporal → bins WOE → 18 variables → regresión logística → PD → política PD < 20%**

La selección metodológica completa se encuentra en:

`01_desarrollo_modelo_logistico.ipynb`

La comparación contra un modelo más complejo se realizará en:

`03_benchmark_catboost.ipynb`

### Resultado de negocio esperado

La regla definida con VALIDATION:

> **Habilitar renovación si PD < 20%**

se mantiene sin modificaciones en OOT y permite:

- aprobación cercana a 73–75%;
- Bad Rate de aprobados alrededor o por debajo del objetivo de 8%;
- concentración de más del 60% de los malos en el grupo rechazado en OOT.
